In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb

# Point to where chroma_db actually got created
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

chroma_client = chromadb.PersistentClient(path="c:/Users/aarus/Context_core/utils/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("contextcore")
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
)

print("Index loaded successfully")

Index loaded successfully


In [10]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

# Using GPT-4o-mini — good balance of quality and cost
# Each ReAct query costs ~$0.001-0.002 at this model
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

print("LLM ready:", Settings.llm.model)

LLM ready: gpt-4o-mini


defining retriver

In [11]:
from llama_index.core.tools import FunctionTool

# Define the retriever
retriever = index.as_retriever(similarity_top_k=3)

def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for macro economic data,
    background concepts, and recent news articles.
    Use this for: interest rates, inflation, GDP, 
    unemployment, yield curves, and general finance/macro/tech concepts.
    """
    nodes = retriever.retrieve(query)
    
    results = []
    for node in nodes:
        source = node.metadata.get("source", "unknown")
        text   = node.text[:300]
        results.append(f"[{source}]: {text}")
    
    return "\n\n".join(results)

# Wrap as a LlamaIndex tool
knowledge_tool = FunctionTool.from_defaults(
    fn=search_knowledge_base,
    name="search_knowledge_base",
    description="""Use this to find US economic indicators (interest rates, 
    CPI, GDP, unemployment), background concepts (what is inflation, yield curve etc),
    and recent news articles. Do NOT use for live breaking news."""
)

# Quick test
result = search_knowledge_base("what is the current federal funds rate?")
print(result)

[FRED]: FRED Data: Federal Funds Rate

In 2024-06, the Federal Funds Rate was 5.33.
In 2024-07, the Federal Funds Rate was 5.33.
In 2024-08, the Federal Funds Rate was 5.33.
In 2024-09, the Federal Funds Rate was 5.13.
In 2024-10, the Federal Funds Rate was 4.83.
In 2024-11, the Federal Funds Rate was 4.64.

[FRED]: FRED Data: Unemployment Rate

In 2024-06, the Unemployment Rate was 4.10.
In 2024-07, the Unemployment Rate was 4.20.
In 2024-08, the Unemployment Rate was 4.20.
In 2024-09, the Unemployment Rate was 4.10.
In 2024-10, the Unemployment Rate was 4.10.
In 2024-11, the Unemployment Rate was 4.20.
In 202

[FRED]: FRED Data: 10-Year Treasury Yield

In 2026-05, the 10-Year Treasury Yield was 4.41.
In 2026-05, the 10-Year Treasury Yield was 4.38.
In 2026-05, the 10-Year Treasury Yield was 4.42.
In 2026-05, the 10-Year Treasury Yield was 4.46.
In 2026-05, the 10-Year Treasury Yield was 4.46.
In 2026-05, the 10-Y


live NewsAPI search

In [12]:
import requests

def search_live_news(query: str) -> str:
    """
    Search for recent news articles on a topic.
    Use this for live, breaking, or recent events that 
    may not be in the knowledge base yet.
    Do NOT use for background concepts or macro indicators.
    """
    response = requests.get(
        "https://newsapi.org/v2/everything",
        params={
            "q": query,
            "language": "en",
            "sortBy": "relevancy",
            "pageSize": 3,
            "apiKey": os.getenv("NEWS_API_KEY")
        }
    )
    
    articles = response.json().get("articles", [])
    
    if not articles:
        return "No recent news found for this query."
    
    results = []
    for a in articles:
        results.append(
            f"[NEWS] {a['title']}\n"
            f"Source: {a['source']['name']} | {a.get('publishedAt','')[:10]}\n"
            f"{a.get('description','')}"
        )
    
    return "\n\n".join(results)

news_tool = FunctionTool.from_defaults(
    fn=search_live_news,
    name="search_live_news",
    description="""Use this to find recent or breaking news articles.
    Use for current events, recent policy decisions, market moves.
    Do NOT use for definitions or historical macro data."""
)

# Quick test
print(search_live_news("Fed interest rate decision 2026"))

[NEWS] The War Against Inflation Would Nearly Be Over If Not for Trump
Source: Jezebel | 2026-05-29
We would be in an entirely different economy if practically anyone other than America’s worst human was president.

[NEWS] A top bank says the Fed must act now to address the highest bond yields since 2007
Source: Business Insider | 2026-05-20
The Fed can't wait until its June meeting to address surging bond yields. Strategists say Fed officials should turn hawkish now to calm markets.

[NEWS] Last week's stock plunge was a 'healthy reset' that sets up a fresh rally, Morgan Stanley says
Source: Business Insider | 2026-06-08
The stock market's brutal sell-off on Friday marks what Morgan Stanley's CIO says is a "healthy reset" that sets the stage for fresh gains.


domain classifier

In [15]:
import re

def classify_domain(headline: str) -> str:
    headline_lower = headline.lower()
    
    # Use word boundaries so "ai" doesn't match "raises"
    def matches(words, text):
        return any(re.search(r'\b' + w + r'\b', text) for w in words)
    
    domains = []
    
    if matches(["fed", "rate", "inflation", "gdp", "unemployment", "recession", "treasury", "monetary"], headline_lower):
        domains.append("macro")
    
    if matches(["stock", "market", "equity", "bond", "bank", "earnings", "valuation", "nasdaq", "s&p"], headline_lower):
        domains.append("finance")
    
    if matches(["chip", "semiconductor", "ai", "tech", "export", "nvidia", "apple", "microsoft", "startup"], headline_lower):
        domains.append("tech")
    
    if len(domains) == 0:
        domains.append("general")
    
    if len(domains) > 1:
        domains.append("cross-domain")
    
    result = f"Domain classification: {', '.join(domains)}"
    return result

# Re-test
print(classify_domain("Fed raises rates as inflation hits 3 year high"))
print(classify_domain("Nvidia chip export ban hits semiconductor stocks"))

Domain classification: macro
Domain classification: tech


In [19]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent(
    tools=[domain_tool, knowledge_tool, news_tool],
    llm=Settings.llm,
    verbose=True,
    max_iterations=10
)

print("Agent ready with 3 tools:")
print("  - classify_domain")
print("  - search_knowledge_base")
print("  - search_live_news")

Agent ready with 3 tools:
  - classify_domain
  - search_knowledge_base
  - search_live_news


In [23]:
response = await agent.run(
    "Fed raises interest rates by 0.25%. What does this mean for tech stocks and inflation?"
)

print("\n=== FINAL ANSWER ===")
print(response)

[tick] add: AgentWorkflowStartEvent(user_msg='Fed raises interest rates by 0.25%. What does this...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Fed raises interest rates by 0.25%. What does this mean for tech stoc...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Fed raises interest rates by 0.25%. What does this mean for tech stoc...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_

In [24]:
# print([m for m in dir(agent) if not m.startswith("_")])